# Protocol B results (patient-level split)
Reads the saved `preds_patient_resplit_seed*.npz` files. No GPU or training needed; it takes seconds. Threshold is chosen on validation only.


In [1]:
import os, glob
import numpy as np
from sklearn.metrics import roc_curve, roc_auc_score, accuracy_score, recall_score, confusion_matrix

H5_DIR = r"C:\CODES__SSD512\Minor_Major_Pneumonia\Major\H5"

files = sorted(glob.glob(os.path.join(H5_DIR, "preds_patient_resplit_seed*.npz")))
print("Found:", [os.path.basename(f) for f in files])

def youden(y, p):
    fv, tv, th = roc_curve(y, p)
    return float(np.clip(th[np.argmax(tv - fv)], 0.05, 0.95))

rows, vals, tests = [], [], []
for f in files:
    d = np.load(f); yt = d["y_test"]
    t = youden(d["y_val"], d["val_preds"])
    b = (d["test_preds"] >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(yt, b, labels=[0, 1]).ravel()
    rows.append([accuracy_score(yt, b), roc_auc_score(yt, d["test_preds"]), recall_score(yt, b), recall_score(yt, b, pos_label=0)])
    vals.append(d["val_preds"]); tests.append(d["test_preds"])
    print(f"{os.path.basename(f):38s} acc={rows[-1][0]:.4f} auc={rows[-1][1]:.4f} sens={rows[-1][2]:.4f} spec={rows[-1][3]:.4f}  t={t:.2f}  TN={tn} FP={fp} FN={fn} TP={tp}")

r = np.array(rows)
print("\nMEAN ± STD over", len(r), "seeds:")
for n, m, s in zip(["accuracy", "AUC", "sensitivity", "specificity"], r.mean(0), r.std(0)):
    print(f"  {n:12s} {m:.4f} ± {s:.4f}")

if len(files) >= 2:
    same = all(np.array_equal(np.load(f)["y_test"], yt) for f in files)
    assert same, "Seeds have different test splits - did SPLIT_SEED change between runs?"
    ev, et = np.mean(vals, 0), np.mean(tests, 0)
    t = youden(d["y_val"], ev); b = (et >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(yt, b, labels=[0, 1]).ravel()
    print(f"\nSEED ENSEMBLE ({len(files)} seeds): acc={accuracy_score(yt, b):.4f} auc={roc_auc_score(yt, et):.4f} "f"sens={tp/(tp+fn):.4f} spec={tn/(tn+fp):.4f}  t={t:.2f}  TN={tn} FP={fp} FN={fn} TP={tp}")


Found: ['preds_patient_resplit_seed123.npz', 'preds_patient_resplit_seed42.npz', 'preds_patient_resplit_seed7.npz']
preds_patient_resplit_seed123.npz      acc=0.9750 auc=0.9954 sens=0.9864 spec=0.9437  t=0.34  TN=151 FP=9 FN=6 TP=434
preds_patient_resplit_seed42.npz       acc=0.9700 auc=0.9958 sens=0.9659 spec=0.9812  t=0.52  TN=157 FP=3 FN=15 TP=425
preds_patient_resplit_seed7.npz        acc=0.9700 auc=0.9950 sens=0.9705 spec=0.9688  t=0.55  TN=155 FP=5 FN=13 TP=427

MEAN ± STD over 3 seeds:
  accuracy     0.9717 ± 0.0024
  AUC          0.9954 ± 0.0003
  sensitivity  0.9742 ± 0.0088
  specificity  0.9646 ± 0.0156

SEED ENSEMBLE (3 seeds): acc=0.9717 auc=0.9960 sens=0.9727 spec=0.9688  t=0.49  TN=155 FP=5 FN=12 TP=428
